In [10]:
import pandas as pd
import numpy as np

class GranularWealthEngine:
    def __init__(self, inputs: dict, step_up_schedule: dict):
        """
        Initializes the model with custom inputs and an explicit manual step-up schedule.
        """
        self.client_name = inputs.get("client_name", "Valued Client")
        self.current_age = inputs.get("current_age", 35)
        self.annual_premium = inputs.get("annual_premium", 150000)
        self.payout_pct = inputs.get("payout_pct", 0.40)
        self.life_cover_multiple = inputs.get("life_cover_multiple", 7)
        self.ppt = inputs.get("ppt", 12)
        self.policy_term = inputs.get("policy_term", 40)
        self.expected_return = inputs.get("expected_return", 0.18)
        self.monthly_swp_target = inputs.get("monthly_swp", 100000)
        self.swp_start_age = inputs.get("swp_start_age", 60)
        
        # User defined structural schedule: { Policy_Year: Absolute_Monthly_Increase_Amount }
        self.step_up_schedule = step_up_schedule
        
        # Derived Base Variables
        self.life_cover = self.annual_premium * self.life_cover_multiple
        self.annual_payout = self.annual_premium * self.payout_pct
        self.base_monthly_sip = round(self.annual_payout / 12, 2)
        self.monthly_rate = self.expected_return / 12
        
    def run_projection(self) -> pd.DataFrame:
        """
        Executes a 40-year projection reacting dynamically to uneven manual inputs.
        """
        records = []
        current_corpus = 0.0
        cumulative_step_up = 0.0
        
        for year in range(1, self.policy_term + 1):
            age = self.current_age + (year - 1)
            
            # 1. Base Premium and Payout rules
            premium_paid = self.annual_premium if year <= self.ppt else 0.0
            insurance_payout = self.annual_payout
            
            # 2. GRANULAR LOGIC: Check if this specific year triggers a fresh manual increase
            year_specific_increment = self.step_up_schedule.get(year, 0.0)
            cumulative_step_up += year_specific_increment
            
            total_monthly_sip = self.base_monthly_sip + cumulative_step_up
            annual_sip_contribution = total_monthly_sip * 12
            
            # 3. Monthly Financial Compounding Loop
            monthly_sip = total_monthly_sip
            monthly_swp = self.monthly_swp_target if age >= self.swp_start_age else 0.0
            
            for month in range(1, 13):
                if current_corpus <= 0:
                    current_corpus = 0.0
                
                # Add monthly investment & compound
                current_corpus += monthly_sip
                current_corpus *= (1 + self.monthly_rate)
                
                # Deduct monthly withdrawal if active
                if current_corpus >= monthly_swp:
                    current_corpus -= monthly_swp
                else:
                    current_corpus = 0.0
            
            # End of Year Corpus Value
            net_corpus = round(current_corpus, 2)
            annual_swp_withdrawn = (self.monthly_swp_target * 12) if age >= self.swp_start_age else 0.0
            
            # 4. Sustainability check
            status = "Corpus Exhausted" if net_corpus <= 0 else "Sustainable"
                
            records.append({
                "Policy Year": year,
                "Age": age,
                "Premium Paid (₹)": premium_paid,
                "Insurance Payout (₹)": insurance_payout,
                "Base Monthly SIP (₹)": self.base_monthly_sip,
                "New Step-Up Added (₹)": year_specific_increment,
                "Total Monthly SIP (₹)": total_monthly_sip,
                "Annual SIP Contribution (₹)": annual_sip_contribution,
                "Annual SWP Withdrawal (₹)": annual_swp_withdrawn,
                "Net End-of-Year Corpus (₹)": net_corpus,
                "Sustainability Flag": status
            })
            
        return pd.DataFrame(records)

    def generate_executive_summary(self, df: pd.DataFrame) -> dict:
        """
        Parses the matrix to generate advisor KPI metrics.
        """
        retirement_row = df[df["Age"] == self.swp_start_age]
        corpus_at_retirement = retirement_row["Net End-of-Year Corpus (₹)"].values[0] if not retirement_row.empty else 0.0
        
        final_corpus = df["Net End-of-Year Corpus (₹)"].iloc[-1]
        
        r_monthly = self.monthly_rate
        n_months = (self.policy_term - (self.swp_start_age - self.current_age)) * 12
        required_corpus = self.monthly_swp_target * ((1 - (1 + r_monthly)**-n_months) / r_monthly) if n_months > 0 else 0.0
            
        exhausted_rows = df[df["Sustainability Flag"] == "Corpus Exhausted"]
        if not exhausted_rows.empty:
            survival_status = "Corpus Exhausted"
            survives_until_year = exhausted_rows["Policy Year"].values[0]
            survives_until_age = exhausted_rows["Age"].values[0]
        else:
            survival_status = "Sustainable"
            survives_until_year = self.policy_term
            survives_until_age = self.current_age + self.policy_term
            
        return {
            "Life Cover Amount": self.life_cover,
            "Monthly Insurance Payout": self.base_monthly_sip,
            "Corpus at SWP Start": corpus_at_retirement,
            "Target Required Corpus": round(required_corpus, 2),
            "Surplus / Shortfall": round(corpus_at_retirement - required_corpus, 2),
            "Final Year 40 Corpus": final_corpus,
            "Sustainability Status": survival_status,
            "Years Corpus Survives": survives_until_year,
            "Age Corpus Exhausted": survives_until_age
        }

    def predict_sustainability_gap(self, df: pd.DataFrame) -> dict:
        """
        Predicts exactly how much corpus is needed at retirement age to make 
        the strategy sustainable, and calculates the specific shortfall if it is not.
        """
        retirement_year_row = df[df["Age"] == self.swp_start_age]
        
        if retirement_year_row.empty:
            actual_corpus_at_start = 0.0
        else:
            target_policy_year = retirement_year_row["Policy Year"].values[0]
            prior_year_row = df[df["Policy Year"] == (target_policy_year - 1)]
            actual_corpus_at_start = prior_year_row["Net End-of-Year Corpus (₹)"].values[0] if not prior_year_row.empty else 0.0

        years_of_withdrawal = self.policy_term - (self.swp_start_age - self.current_age)
        total_withdrawal_months = max(0, years_of_withdrawal * 12)
        
        r = self.monthly_rate
        if r > 0 and total_withdrawal_months > 0:
            factor = (1 - (1 + r) ** -total_withdrawal_months) / r
            exact_target_needed = (self.monthly_swp_target * factor) / (1 + r)
        else:
            exact_target_needed = 0.0
            
        exact_target_needed = round(exact_target_needed, 2)
        shortfall = round(max(0.0, exact_target_needed - actual_corpus_at_start), 2)
        is_sustainable = shortfall == 0.0

        return {
            "Total Withdrawal Months": total_withdrawal_months,
            "Actual Projected Corpus": actual_corpus_at_start,
            "Target Needed": exact_target_needed,
            "Shortfall": shortfall,
            "Is Sustainable": "YES" if is_sustainable else "NO"
        }


class DeficitBridgeEngine:
    def __init__(self, target_gap: float, current_age: int, target_age: int, expected_return: float):
        """
        Calculates Month-on-Month strategy tracking to systematically bridge gaps.
        """
        self.target_gap = target_gap
        self.current_age = current_age
        self.target_age = target_age
        self.monthly_rate = expected_return / 12
        self.total_months = (target_age - current_age) * 12
        
        # Calculate precise required additional MoM investment using Annuity Due logic
        r = self.monthly_rate
        n = self.total_months
        annuity_factor = (((1 + r) ** n - 1) / r) * (1 + r)
        self.required_monthly_investment = round(target_gap / annuity_factor, 2)

    def generate_mom_ledger(self) -> pd.DataFrame:
        records = []
        running_corpus = 0.0
        
        for month in range(1, self.total_months + 1):
            current_policy_year = ((month - 1) // 12) + 1
            calculated_age = self.current_age + ((month - 1) // 12)
            month_of_year = ((month - 1) % 12) + 1
            
            opening_balance = running_corpus
            running_corpus += self.required_monthly_investment
            running_corpus *= (1 + self.monthly_rate)
            
            records.append({
                "Total Month": month,
                "Policy Year": current_policy_year,
                "Age": calculated_age,
                "Month of Year": month_of_year,
                "Opening Balance (₹)": round(opening_balance, 2),
                "MoM Addition (₹)": self.required_monthly_investment,
                "Interest Earned (₹)": round(running_corpus - opening_balance - self.required_monthly_investment, 2),
                "Closing Balance (₹)": round(running_corpus, 2)
            })
            
        return pd.DataFrame(records)


# ════════════════════════════════════
# SCENARIO TESTING SETUP
# ════════════════════════════════════

if __name__ == "__main__":
    
    # Core Structural Planning Inputs
    advisor_inputs = {
        "client_name": "Aditya Sharma",
        "current_age": 50,
        "annual_premium": 150000,
        "payout_pct": 0.40,
        "life_cover_multiple": 7,
        "ppt": 12,
        "policy_term": 40,
        "expected_return": 0.18, 
        "monthly_swp": 100000,   
        "swp_start_age": 60      
    }

    custom_schedule = {
        
        
    }

    # 1. Fire Main Base Model Projections
    engine = GranularWealthEngine(advisor_inputs, custom_schedule)
    projection_matrix = engine.run_projection()
    summary = engine.generate_executive_summary(projection_matrix)
    gap_analysis = engine.predict_sustainability_gap(projection_matrix)

    # 2. Fire MoM Gap Resolution Sub-Engine
    GAP_AMOUNT = 7452095.19  # Explicit shortfall requirement requested
    bridge_calculator = DeficitBridgeEngine(GAP_AMOUNT, engine.current_age, engine.swp_start_age, engine.expected_return)
    mom_matrix = bridge_calculator.generate_mom_ledger()

    # Maximize layout context width to prevent string column wrapping in terminal windows
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 1000)
    pd.set_option('display.max_rows', 250)  # Expand row buffer to accommodate full MoM matrix printing

    # Print Executive Summary Panel
    print(f"\n" + "═"*75)
    print(f" WEALTH PLANNER EXECUTIVE DASHBOARD: {engine.client_name.upper()}")
    print("═"*75)
    print(f" Client Current Age              : {engine.current_age} years old")
    print(f" Target Retirement SWP Age       : {engine.swp_start_age} years old")
    print(f" Initial Monthly Base Payout     : ₹{summary['Monthly Insurance Payout']:,.2f}")
    print(f" Client Custom Schedule Rules    : {custom_schedule}")
    print(f" Total Life Cover Provided       : ₹{summary['Life Cover Amount']:,.2f}")
    print(f" Capital Balance at Age 60       : ₹{summary['Corpus at SWP Start']:,.2f}")
    print(f" Required Balance Target for SWP  : ₹{summary['Target Required Corpus']:,.2f}")
    print(f" Calculated Surplus / Shortfall  : ₹{summary['Surplus / Shortfall']:,.2f}")
    print(f" Terminal Portfolio Value (Yr 40): ₹{summary['Final Year 40 Corpus']:,.2f}")
    print(f" Strategy Sustainability Status  : {summary['Sustainability Status'].upper()}")
    
    if summary['Sustainability Status'] == "Corpus Exhausted":
        print(f" 🚨 ALERT: Running out of cash in Policy Year {summary['Years Corpus Survives']} (Age {summary['Age Corpus Exhausted']})")
        print(f" 🎯 STRATEGY FIX: Add ₹{bridge_calculator.required_monthly_investment:,.2f}/month to bridge the ₹{GAP_AMOUNT:,.2f} gap.")
    print("═"*75 + "\n")

    # Print 40-Year Annual Audit Ledger Matrix
    print(">>> COMPLETE 40-YEAR AUDIT LEDGER MATRIX:")
    print("═"*115)
    all_cols = [
        "Policy Year", "Age", "Premium Paid (₹)", "Insurance Payout (₹)", 
        "Base Monthly SIP (₹)", "New Step-Up Added (₹)", "Total Monthly SIP (₹)", 
        "Annual SIP Contribution (₹)", "Annual SWP Withdrawal (₹)",
        "Net End-of-Year Corpus (₹)", "Sustainability Flag"
    ]
    print(projection_matrix[all_cols].to_string(index=False))
    print("═"*115 + "\n")

    # Print Comprehensive Month-on-Month Accumulation Ledger
    print(f">>> FULL MONTH-ON-MONTH (MoM) DEFICIT BRIDGE LEDGER (To Reach Target: ₹{GAP_AMOUNT:,.2f})")
    print("═"*115)
    columns_mom = ["Total Month", "Policy Year", "Age", "Month of Year", "Opening Balance (₹)", "MoM Addition (₹)", "Interest Earned (₹)", "Closing Balance (₹)"]
    print(mom_matrix[columns_mom].to_string(index=False))
    print("═"*115)


═══════════════════════════════════════════════════════════════════════════
 WEALTH PLANNER EXECUTIVE DASHBOARD: ADITYA SHARMA
═══════════════════════════════════════════════════════════════════════════
 Client Current Age              : 50 years old
 Target Retirement SWP Age       : 60 years old
 Initial Monthly Base Payout     : ₹5,000.00
 Client Custom Schedule Rules    : {}
 Total Life Cover Provided       : ₹1,050,000.00
 Capital Balance at Age 60       : ₹772,240.98
 Required Balance Target for SWP  : ₹6,635,324.17
 Calculated Surplus / Shortfall  : ₹-5,863,083.19
 Terminal Portfolio Value (Yr 40): ₹0.00
 Strategy Sustainability Status  : CORPUS EXHAUSTED
 🚨 ALERT: Running out of cash in Policy Year 12 (Age 61)
 🎯 STRATEGY FIX: Add ₹22,161.87/month to bridge the ₹7,452,095.19 gap.
═══════════════════════════════════════════════════════════════════════════

>>> COMPLETE 40-YEAR AUDIT LEDGER MATRIX:
═════════════════════════════════════════════════════════════════════════════════